# LiC Eval Subset

In [15]:
import json
import random
from pathlib import Path

from datasets import load_dataset

/Users/matthewho/miniconda3/envs/collabmem/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/matthewho/Documents/research/ctx_editor


In [5]:
def read_json(path):
    with open(path, "r") as f:
        data = json.load(f)
    return data

In [3]:
lic_path = REPO_ROOT / "data/sharded_instructions_600.json"

In [ ]:
lic_data = read_json(lic_path)

## create subset

In [8]:
task_subset = ["math", "code", "actions", "database"]

In [9]:
subset_size_per_task = 30

In [11]:
samples_per_task = {task: [] for task in task_subset}
for sample in lic_data:
    task = sample["task"]
    if task in task_subset:
        samples_per_task[task].append(sample)

In [12]:
print({task: len(samples) for task, samples in samples_per_task.items()})

{'math': 103, 'code': 100, 'actions': 105, 'database': 107}


In [13]:
# set a seed
random.seed(42)
eval_subset = []
for task, samples in samples_per_task.items():
    selected_samples = random.sample(samples, subset_size_per_task)
    eval_subset.extend(selected_samples)
print(f"Total eval subset size: {len(eval_subset)}")

Total eval subset size: 120


In [14]:
# write to file
subset30_path = REPO_ROOT / "data/lic_subset30.json"
with open(subset30_path, "w") as f:
    json.dump(eval_subset, f, indent=2)
print(f"Wrote eval subset to {subset30_path}")

Wrote eval subset to /Users/matthewho/Documents/research/ctx_editor/data/lic_subset30.json


## add standard keys for reflection
on reflection we want to be able to look at
1. the single turn specification of the question
2. the ground truth answer (if available)

Most of this information is already in the data, but they all have different keys.
Let's standardize them

In [ ]:
# humaneval dataset
he_dataset = load_dataset("openai/openai_humaneval")

In [ ]:
# only for answers
lcb_exec2_dataset = load_dataset("livecodebench/execution-v2")
lcb_exec2_dict = {item["question_id"]: item for item in lcb_exec2_dataset["test"]}

In [ ]:
def add_full_spec_qa(data: list) -> None:
    for item in data:
        task = item["task"]
        if task not in task_subset:
            continue

        if task == "math":
            item["full_spec_q"] = item["question"]
            item["ground_truth_a"] = item["answer"]
        elif task == "code":
            # task id options:
            # - sharded-HumanEval/{number}
            # - sharded-livecodebench/{number}
            if item["task_id"].startswith("sharded-HumanEval/"):
                item["full_spec_q"] = item.get("prompt", None)
                number = int(item["task_id"].split("/")[1])
                item["ground_truth_a"] = he_dataset["test"][number][
                    "canonical_solution"
                ]
            else:
                item["full_spec_q"] = item.get("question_content", None)
                # NOTE [2026.01.27]
                # - we can use execution-v2 dataset to get ground truth answers
                # - there's not a unique answer per question, the execution-v2 dataset contains multiple per question_id
                # - we can just pick any one of them (e.g. first or last)
                # - deferring for now
                item["ground_truth_a"] = None
                if item["full_spec_q"] is None:
                    print(
                        f"livecodebench task id: {item['task_id']}, no full spec question available"
                    )
        elif task == "actions":
            item["full_spec_q"] = item["fully_specified_question"][0][0]["content"]
            item["ground_truth_a"] = item["reference_answer"]
        elif task == "database":
            item["full_spec_q"] = item["fully_specified_question"]
            item["ground_truth_a"] = item["reference_sql"]
        else:
            print(f"skipping {item['task_id']}")
            item["full_spec_q"] = None
            item["ground_truth_a"] = None

In [18]:
add_full_spec_qa(eval_subset)

In [20]:
# write to file
output_path = REPO_ROOT / "data/lic_eval_subset.json"
with open(output_path, "w") as f:
    json.dump(eval_subset, f, indent=2)
print(f"Wrote eval subset with full spec QA to {output_path}")

Wrote eval subset with full spec QA to /Users/matthewho/Documents/research/ctx_editor/data/lic_eval_subset.json


## check statistics

In [28]:
# check split between humaneval and livecodebench
he_count = 0
lcb_count = 0
for item in eval_subset:
    if item["task_id"].startswith("sharded-HumanEval/"):
        he_count += 1
    elif item["task_id"].startswith("sharded-livecodebench/"):
        lcb_count += 1
        print(f"LiveCodeBench item: {item['task_id']}")

print(f"HumanEval count: {he_count}, LiveCodeBench count: {lcb_count}")

LiveCodeBench item: sharded-livecodebench/2979
LiveCodeBench item: sharded-livecodebench/2892
LiveCodeBench item: sharded-livecodebench/2792
LiveCodeBench item: sharded-livecodebench/2954
LiveCodeBench item: sharded-livecodebench/2755
LiveCodeBench item: sharded-livecodebench/2727
LiveCodeBench item: sharded-livecodebench/2872
LiveCodeBench item: sharded-livecodebench/2812
LiveCodeBench item: sharded-livecodebench/2847
LiveCodeBench item: sharded-livecodebench/2888
LiveCodeBench item: sharded-livecodebench/2855
LiveCodeBench item: sharded-livecodebench/2728
LiveCodeBench item: sharded-livecodebench/2878
HumanEval count: 17, LiveCodeBench count: 13


In [30]:
# check out split in original full dataset
he_full_count = 0
lcb_full_count = 0
for item in lic_data:
    if item["task_id"].startswith("sharded-HumanEval/"):
        he_full_count += 1
    elif item["task_id"].startswith("sharded-livecodebench/"):
        lcb_full_count += 1
print(f"Full dataset - HumanEval count: {he_full_count}, LiveCodeBench count: {lcb_full_count}")

Full dataset - HumanEval count: 45, LiveCodeBench count: 55


## check lcb execution

In [ ]:
lcb_exec2_dataset = load_dataset("livecodebench/execution-v2")
lcb_exec2_dict = {item["question_id"]: item for item in lcb_exec2_dataset["test"]}

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 77b4a6b0-04b4-46fe-b79f-e8a1cf54d69f)')' thrown while requesting HEAD https://huggingface.co/datasets/livecodebench/execution-v2/resolve/main/README.md
Retrying in 1s [Retry 1/5].
Generating test split: 100%|██████████| 479/479 [00:00<00:00, 73455.14 examples/s]


In [31]:
lcb_exec2_dict = {item["question_id"]: item for item in lcb_exec2_dataset["test"]}

In [ ]:
lic_subset_dict = {item["task_id"]: item for item in eval_subset}

In [46]:
len(lcb_exec2_dict)

92

In [50]:
dummy_lcb_lic_row = lic_subset_dict["sharded-livecodebench/2979"]
list(dummy_lcb_lic_row.keys())

['question_title',
 'question_content',
 'platform',
 'question_id',
 'contest_id',
 'contest_date',
 'starter_code',
 'difficulty',
 'public_test_cases',
 'private_test_cases',
 'metadata',
 'task_id',
 'source',
 'shards',
 'task',
 'full_spec_q',
 'ground_truth_a']

In [58]:
print(dummy_lcb_lic_row["source"])
print(dummy_lcb_lic_row["question_content"])

lcb_medium
You are given an integer n representing the number of houses on a number line, numbered from 0 to n - 1.
Additionally, you are given a 2D integer array offers where offers[i] = [start_i, end_i, gold_i], indicating that i^th buyer wants to buy all the houses from start_i to end_i for gold_i amount of gold.
As a salesman, your goal is to maximize your earnings by strategically selecting and selling houses to buyers.
Return the maximum amount of gold you can earn.
Note that different buyers can't buy the same house, and some houses may remain unsold.
 
Example 1:

Input: n = 5, offers = [[0,0,1],[0,2,2],[1,3,2]]
Output: 3
Explanation: There are 5 houses numbered from 0 to 4 and there are 3 purchase offers.
We sell houses in the range [0,0] to 1^st buyer for 1 gold and houses in the range [1,3] to 3^rd buyer for 2 golds.
It can be proven that 3 is the maximum amount of gold we can achieve.

Example 2:

Input: n = 5, offers = [[0,0,1],[0,2,10],[1,3,2]]
Output: 10
Explanation: The

In [41]:
lcb_exec2_dataset

DatasetDict({
    test: Dataset({
        features: ['question_id', 'id', 'function_name', 'code', 'input', 'output', 'numsteps', 'problem_id', 'contest_id', 'contest_date', 'difficulty'],
        num_rows: 479
    })
})

In [45]:
# check if 'question_id' column is unique
question_ids = [item["question_id"] for item in lcb_exec2_dataset["test"]]
unique_question_ids = set(question_ids)
print(f"Total question IDs: {len(question_ids)}, Unique question IDs: {len(unique_question_ids)}")

Total question IDs: 479, Unique question IDs: 92


In [43]:
print(lcb_exec2_dataset["test"][0])

{'question_id': 2777, 'id': 'sample_0', 'function_name': 'distinctDifferenceArray', 'code': 'def distinctDifferenceArray(a: List[int]) -> List[int]:\n    return [len(set(a[:i+1]))-len(set(a[i+1:]))for i in range(len(a))]', 'input': 'distinctDifferenceArray(a = [1, 2, 3, 4, 5])', 'output': '[-3, -1, 1, 3, 5]', 'numsteps': 678, 'problem_id': [0, 2, 0], 'contest_id': 'weekly-contest-344', 'contest_date': datetime.datetime(2023, 5, 7, 0, 0), 'difficulty': 'easy'}


In [35]:
print(lcb_exec2_dict[2979]["code"])

def maximizeTheProfit(N: int, offers: List[List[int]]) -> int:
    best = [0] * (N + 1)
    
    prev = collections.defaultdict(list)
    
    for a, b, w in offers:
        prev[b].append((a - 1, w))
        
    for i in range(N):
        best[i + 1] = max(best[i], best[i + 1])
        for p, w in prev[i]:
            best[i + 1] = max(best[i + 1], best[p + 1] + w)
    # print(best)
    return best[N]


In [59]:
# leetcode dataset
leetcode_dataset = load_dataset("greengerong/leetcode")

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: d90efe75-32c8-4af7-a3c8-ee6995c8555a)')' thrown while requesting HEAD https://huggingface.co/datasets/greengerong/leetcode/resolve/main/README.md
Retrying in 1s [Retry 1/5].
Generating train split: 100%|██████████| 2360/2360 [00:00<00:00, 47163.16 examples/s]


In [60]:
leetcode_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'slug', 'title', 'difficulty', 'content', 'java', 'c++', 'python', 'javascript'],
        num_rows: 2360
    })
})

In [70]:
leetcode_dataset["train"][2359]

{'id': 2612,
 'slug': 'minimum-reverse-operations',
 'title': 'Minimum Reverse Operations',
 'difficulty': 'Hard',
 'content': "You are given an integer `n` and an integer `p` in the range `[0, n - 1]`. Representing a **0-indexed** array `arr` of length `n` where all positions are set to `0`'s, except position `p` which is set to `1`.\n\nYou are also given an integer array `banned` containing some positions from the array. For the **i****th** position in `banned`, `arr[banned[i]] = 0`, and `banned[i] != p`.\n\nYou can perform **multiple** operations on `arr`. In an operation, you can choose a **subarray** with size `k` and **reverse** the subarray. However, the `1` in `arr` should never go to any of the positions in `banned`. In other words, after each operation `arr[banned[i]]` **remains** `0`.\n\n_Return an array_ `ans` _where_ _for each_ `i` _from_ `[0, n - 1]`, `ans[i]` _is the **minimum** number of reverse operations needed to bring the_ `1` _to position_ `i` _in arr_, _or_ `-1` _

In [75]:
lcb_genlite = load_dataset("livecodebench/code_generation_lite")

RuntimeError: Dataset scripts are no longer supported, but found code_generation_lite.py